# Exploratory data analysis (interactive)

Interactive tour of the Sentinel-1 dataset with Plotly: hover for values, zoom, and a month slider on the radar. Run after the data is exported. Reads from `OUT`. Each figure also saves as a self-contained HTML you can open or embed.

In [ ]:
# 1. Setup
import os, sys, glob
for p in ['/home/jovyan/vault/sar-lulc', os.getcwd(), os.path.dirname(os.getcwd()), os.path.dirname(os.path.dirname(os.getcwd()))]:
    if os.path.isdir(os.path.join(p, 'src', 'lulc')):
        os.chdir(p); break
else:
    raise RuntimeError("could not find the repo root (a folder with src/lulc). cd into sar-lulc and rerun.")
sys.path.insert(0, os.path.abspath('src'))
print('working dir:', os.getcwd())
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from lulc.config import load_config
from lulc.data import load_store_ym
from lulc.quant import dequantize_db, NODATA_CODE
from lulc.classes import CLASS_NAMES, CLASS_COLORS, remap_dynamic_world

def month_sequence(year, month=1, n=12):
    i = year * 12 + month - 1
    return [((i + k) // 12, (i + k) % 12 + 1) for k in range(n)]
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
COLORS = dict(zip(CLASS_NAMES, CLASS_COLORS))

OUT = os.environ.get('LULC_OUT', '/home/jovyan/vault/lulc/raw')   # <- your export folder
YEAR, REGIONS = load_config('configs/regions.yaml')
HAVE = [r for r in REGIONS if glob.glob(f'{OUT}/{r.key}/{YEAR}-*.npz')]
print(f'{len(HAVE)} of {len(REGIONS)} regions exported under {OUT}')
def save(fig, name): fig.write_html(f'{name}.html'); fig.show()

## 2. Where the sites are
Hover a point for its name and split. Training is Europe-heavy; test and out-of-distribution sites spread worldwide.

In [ ]:
# 2. Site map (interactive, needs no exported data)
import pandas as pd
df = pd.DataFrame([{'region': r.key, 'name': r.name, 'split': r.split, 'lat': r.lat, 'lon': r.lon,
                    'size_km': 2*r.half_size_km} for r in REGIONS])
fig = px.scatter_geo(df, lat='lat', lon='lon', color='split', hover_name='name',
                     hover_data={'region':True,'split':True,'lat':':.2f','lon':':.2f'},
                     projection='natural earth', title=f'{len(REGIONS)} sites by split')
fig.update_traces(marker=dict(size=9, line=dict(width=0.5, color='black')))
fig.update_layout(height=520, legend_title='split')
save(fig, 'eda_site_map')

## 3. Class distribution in the training data
Hover for the exact share. Built-up and cropland dominate; Grass/shrub, Bare and Water are small, which is why class weighting matters.

In [ ]:
# 3. Class pixel counts across the TRAINING regions
counts = np.zeros(6, np.int64)
train = [r for r in HAVE if r.split == 'train']
for i, r in enumerate(train):
    st = load_store_ym(OUT, r.key, month_sequence(YEAR), r.split)
    lab = st.label[st.present]
    for k in range(6): counts[k] += int((lab == k).sum())
    print(f'  {i+1}/{len(train)} {r.key}      ', end='\r')
share = 100 * counts / counts.sum()
fig = go.Figure(go.Bar(x=list(CLASS_NAMES), y=share, marker_color=CLASS_COLORS,
                       text=[f'{v:.1f}%' for v in share], textposition='outside',
                       hovertemplate='%{x}<br>%{y:.2f}% of labelled pixels<extra></extra>'))
fig.update_layout(title='Class distribution (training regions)', yaxis_title='% of labelled training pixels', height=430)
save(fig, 'eda_class_distribution')

## 4. Radar backscatter per class
VV and VH in dB per class. Hover the boxes for the quartiles. Water is darkest, Built-up brightest; the vegetation classes overlap, which is the core difficulty of radar-only mapping.

In [ ]:
# 4. VV / VH distributions per class (sampled)
rng = np.random.default_rng(0); PER = 40000
vv_by = [[] for _ in range(6)]; vh_by = [[] for _ in range(6)]
for i, r in enumerate(train):
    st = load_store_ym(OUT, r.key, month_sequence(YEAR), r.split)
    t = int(np.flatnonzero(st.present)[0])
    vv, vh, lab = dequantize_db(st.s1[t,0]), dequantize_db(st.s1[t,1]), st.label[t]
    idx = np.flatnonzero((st.s1[t,0] != NODATA_CODE) & (lab != 255))
    if idx.size > PER: idx = rng.choice(idx, PER, replace=False)
    fv, fh, fl = vv.ravel()[idx], vh.ravel()[idx], lab.ravel()[idx]
    for k in range(6): vv_by[k].append(fv[fl==k]); vh_by[k].append(fh[fl==k])
    print(f'  {i+1}/{len(train)} {r.key}      ', end='\r')
fig = make_subplots(rows=1, cols=2, subplot_titles=('VV (dB)', 'VH (dB)'))
for k, name in enumerate(CLASS_NAMES):
    fig.add_trace(go.Box(y=np.concatenate(vv_by[k]) if vv_by[k] else [], name=name, marker_color=CLASS_COLORS[k], boxpoints=False, showlegend=False), 1, 1)
    fig.add_trace(go.Box(y=np.concatenate(vh_by[k]) if vh_by[k] else [], name=name, marker_color=CLASS_COLORS[k], boxpoints=False, showlegend=False), 1, 2)
fig.update_layout(title='Backscatter by class', height=470)
save(fig, 'eda_backscatter_by_class')

## 5. Seasonal signal
Mean VV and VH per month, averaged over the training regions. This swing is what the 3-month window and the yearly statistics use, and what makes a monthly map change without real land-use change.

In [ ]:
# 5. Monthly mean backscatter (seasonality)
mv = np.full((len(train),12), np.nan); mh = np.full((len(train),12), np.nan)
for i, r in enumerate(train):
    st = load_store_ym(OUT, r.key, month_sequence(YEAR), r.split)
    for t in range(12):
        if st.present[t]:
            ok = st.s1[t,0] != NODATA_CODE
            if ok.any(): mv[i,t]=dequantize_db(st.s1[t,0])[ok].mean(); mh[i,t]=dequantize_db(st.s1[t,1])[ok].mean()
    print(f'  {i+1}/{len(train)} {r.key}      ', end='\r')
fig = go.Figure()
fig.add_trace(go.Scatter(x=MONTHS, y=np.nanmean(mv,0), mode='lines+markers', name='VV', line=dict(color='#222')))
fig.add_trace(go.Scatter(x=MONTHS, y=np.nanmean(mh,0), mode='lines+markers', name='VH', line=dict(color='#888', dash='dash')))
fig.update_layout(title='Seasonal backscatter (mean over training regions)', yaxis_title='mean backscatter (dB)', height=420)
save(fig, 'eda_seasonality')

## 6. One region, month by month
Press play or drag the slider to step through the 12 monthly radar composites (false colour: R=VV, G=VH, B=VV-VH). Zoom with the mouse. Change `REGION` to any exported site.

In [ ]:
# 6. Example region: 12 months of radar (interactive slider) + the label
REGION = 'munich'
st = load_store_ym(OUT, REGION, month_sequence(YEAR), 'train')
def fc(t):
    vv, vh = dequantize_db(st.s1[t,0]), dequantize_db(st.s1[t,1])
    rgb = np.stack([vv, vh, vv-vh], -1)
    for b,(lo,hi) in enumerate([(-20,0),(-27,-7),(2,15)]): rgb[...,b] = np.clip((rgb[...,b]-lo)/(hi-lo),0,1)
    return (np.nan_to_num(rgb)*255).astype(np.uint8)
stack = np.stack([fc(t) for t in range(12)])
fig = px.imshow(stack, animation_frame=0, title=f'{REGION}: Sentinel-1 false colour (R=VV, G=VH, B=VV-VH)')
for i,f in enumerate(fig.frames): f.layout.title = f'{REGION}: {MONTHS[i]}'
fig.update_layout(height=620, coloraxis_showscale=False)
save(fig, f'eda_{REGION}_radar')
# annual label as a coloured map
annual = np.stack([(st.label==k).sum(0) for k in range(6)]).argmax(0)
annual[(st.label!=255).sum(0)==0] = 255
pal = np.array([[int(c[i:i+2],16) for i in (1,3,5)] for c in CLASS_COLORS] + [[0,0,0]], np.uint8)
rgb = pal[np.where(annual==255, 6, annual)]
figl = px.imshow(rgb, title=f'{REGION}: Dynamic World label (annual mode)')
figl.update_layout(height=520)
save(figl, f'eda_{REGION}_label')

## 7. Radar coverage (no-data)
Share of no-data radar pixels per region, worst first. Some months or regions have gaps where the chosen orbit direction has no scene.

In [ ]:
# 7. No-data share per region
rows=[]
for r in HAVE:
    st = load_store_ym(OUT, r.key, month_sequence(YEAR), r.split)
    present = st.s1[st.present]
    rows.append((r.key, r.split, float((present[:,0]==NODATA_CODE).mean())*100))
    print(f'  {r.key}      ', end='\r')
rows.sort(key=lambda x: x[2]); worst = rows[-20:]
fig = go.Figure(go.Bar(x=[v for *_,v in worst], y=[f'{k} ({s})' for k,s,_ in worst], orientation='h',
                       marker_color='#888', hovertemplate='%{y}<br>%{x:.1f}% no-data<extra></extra>'))
fig.update_layout(title='Radar coverage gaps (worst 20 regions)', xaxis_title='% no-data radar pixels', height=max(400, 22*len(worst)))
save(fig, 'eda_nodata')
print('regions >5% no-data:', [(k,round(v,1)) for k,_,v in rows if v>5])